
# ROSMAP AC Phenotype data 

- `input`:    
    1. today data from xqtl-protocal : for alignment pipeline testing
    2. 1141 ROSMAP AC junc files from BU (the upstream bam files have not been precessed with WASP yet)


- `output`:
    1. splicing events from leafcutter2
    2. the phenotype data for association anlaysis


## 0. data preparation

For this part, I am using the toydata for the testing of new STAR alignment process.... 

### RNA Seq Alignment
see [previous work ](https://cumc.github.io/xqtl-pipeline/code/molecular_phenotypes/bulk_expression.html)

### Perform data quality summary via `fastqc`
see [previous work ](https://cumc.github.io/xqtl-pipeline/code/molecular_phenotypes/bulk_expression.html)

### Cut adaptor (Optional)
see [previous work ](https://cumc.github.io/xqtl-pipeline/code/molecular_phenotypes/bulk_expression.html)
This step will trim the fastq file to remove the adaptor. It is optional because the fastq in the protocol data folders are converted from bam file and are already without adaptors.


### Read alignment via STAR with WASP and QC via Picard (toydata for pipeline test)

## 1. Phenotype data: Leaf_cutter2

**For this part, I am using the junc files from BU with STAR alignment without WASP**

In [3]:
mkdir -p /home/rf2872/Work/leaf_cutter2/ROSMAP_AC_2024/junc_files && cd /home/rf2872/Work/leaf_cutter2/ROSMAP_AC_2024
ln -s /mnt/vast/hpc/csg/ftp_lisanwanglab_sync/ftp_fgc_xqtl/projects/splicing/BU/ROSMAP_AC/leafcutter/*junc ./junc_files/
ln -s ~/codes/xqtl-pipeline/pipeline/ ./


ln: failed to create symbolic link './pipeline': File exists


: 1

### prepare whole junc file for analysis



In [14]:
sos run  pipeline/splicing_normalization.ipynb Junc_list \
    --junc_path ~/Work/leaf_cutter2/ROSMAP_AC_2024/junc_files \
    --dataset ROSMAP_AC \
    --file_suffix out_wasp_qc.md.junc


INFO: Running Junc_list: 
INFO: Junc_list is completed.
INFO: Junc_list output:   /mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_AC_2024/output/leafcutter2/ROSMAP_AC_intron_usage_perind.junc
INFO: Workflow Junc_list (ID=wcefb1aa389fbdd6d) is executed successfully with 1 completed step.


### (Optional) remove `bam` suffix from sample id 

In [7]:
sed 's/.bam//g' /mnt/vast/hpc/csg/ftp_lisanwanglab_sync/ftp_fgc_xqtl/projects/rna-seq/BU/ROSMAP_AC/sample_map.txt > sample_map_mod.txt

### prepare sample list for analysis

`sample_id` is the ID for RNAseq/bam file preffix, and `participant_id` is the ID for WGS/geno file preffix. Here I use the sample_lookup file from Hao, but I need to cut it to the same length as my files first.

In [13]:
sos run pipeline/splicing_normalization.ipynb Jointcall_samples \
    --sample_table sample_map_mod.txt  \
    --junc_list output/leafcutter2/ROSMAP_AC_intron_usage_perind.junc \
    --file_suffix out_wasp_qc.md.junc


INFO: Running Jointcall_samples: 
INFO: Jointcall_samples is completed.
INFO: Jointcall_samples output:   /mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_AC_2024/output/leafcutter2/sample_map_mod.txt.rnaseq
INFO: Workflow Jointcall_samples (ID=w1685f2ba3e7d624a) is executed successfully with 1 completed step.


### run the leafcutter2 script to generate leafcutter2 outputs:
*  `input`: the "*refined_noisy" out file that includes noisy introns from previous step, -N annotation files `gencode_v43_plus_v37_productive.intron_by_transcript_BEDlike.txt.gz` providing 'functional' or 'productive' info from author,  the "*intron_usage_perind.junc " file from previous and previous step 
*  `output`: different type of leafcutter2_perind.counts.* files. while leafcutter2_perind.counts.noise_by_intron.gz has 5 columns splited by ":" in chrom. 

    - {out_prefix}_perind.counts.noise.gz: output functional introns (intact), and 
                  noisy introns. Note the start and end coordinates of noisy introns are recalibrated
                  to the min(starts) and max(ends) of all functional introns within cluster.
    - {out_prefix}_perind_numers.counts.noise.gz: same as above, except write numerators.
    - {out_prefix}_perind.counts.noise_by_intron.gz: same as the first output, except here
                  noisy introns' coordinates are kept as their original coordinates.  

In [15]:
nohup python /home/rf2872/codes/leafcutter2/scripts/leafcutter2_regtools.py \
    -A ~/data/ref_data_Ru/gencode.v45.basic.annotation.gtf.gz \
    -j output/leafcutter2/ROSMAP_AC_intron_usage_perind.junc \
    -G /mnt/vast/hpc/csg/rf2872/data/ref_data_Ru/hg38.fa.gz\
    -o ROSMAP_AC \
    -r output/leafcutter2 &> leafcutter2.log &

[1] 63343


### (Optional) Filtering the output of leafcutter2

### QC and Normalization of leafCutter2 outputs
*  `input`: the "_intron_usage_perind.counts.gz" file from previous step # here I use _perind.counts.noise.gz, which has the same format with leaf_cutter
*  `output`: QC'd and normalized phenotype table end with "qqnorm.txt"
Be noted that the `ratio` file to be fed into the leafcutter_norm are the one without `number` tag in its filename. 

In [ ]:
sos run  ~/codes/xqtl-pipeline/pipeline/splicing_normalization.ipynb leafcutter_norm \
    --cwd output/leafcutter2/ \
    --ratios output/leafcutter2/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz \
    --container oras://ghcr.io/cumc/leafcutter_apptainer:latest 

[1]+  Done                    nohup python /home/rf2872/codes/leafcutter2/scripts/leafcutter2_regtools.py -A ~/data/ref_data_Ru/gencode.v45.basic.annotation.gtf.gz -j output/leafcutter2/ROSMAP_AC_intron_usage_perind.junc -G /mnt/vast/hpc/csg/rf2872/data/ref_data_Ru/hg38.fa.gz -o ROSMAP_AC -r output/leafcutter2 &> leafcutter2.log
INFO: Running leafcutter_norm_1: 


In [3]:
cd /home/rf2872/Work/leaf_cutter2/ROSMAP_AC_2024
sos run  ~/codes/xqtl-pipeline/pipeline/splicing_normalization.ipynb leafcutter_norm \
    --cwd output/leafcutter_test/ \
    --ratios output/leafcutter2/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz \
    --container oras://ghcr.io/cumc/leafcutter_apptainer:latest 

INFO: Running leafcutter_norm_1: 
INFO: leafcutter_norm_1 is completed.
INFO: leafcutter_norm_1 output:   output/leafcutter2/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_phenotype_file_list.txt
INFO: Running leafcutter_norm_2: 
Analyzing autosomes 1 to 22...
INFO: leafcutter_norm_2 is completed.
INFO: leafcutter_norm_2 output:   output/leafcutter2/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.txt
INFO: Running leafcutter_norm_3: 
INFO: leafcutter_norm_3 is completed.
INFO: leafcutter_norm_3 output:   output/leafcutter2/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.txt
INFO: Workflow leafcutter_norm (ID=w60d4447546564c48) is executed successfully with 3 completed steps.


### Imputation


In [ ]:
sos run pipeline/phenotype_imputation.ipynb EBMF \
    --phenoFile output/leafcutter2/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.txt \
    --cwd output/normalize_impute \
    --prior ebnm_point_laplace --varType 1 \
    --container oras://ghcr.io/cumc/factor_analysis_apptainer:latest \
    --mem 40G \
    --numThreads 20 \
    --walltime 100h \
    -c ~/env_files/csg.yml 

### Post-process of leafcutter outputs for them to be TensorQTL ready
*  `input`: output of the previous two steps and the gtf file.
*  `output`: a file in bed format end with "formated.bed.gz" 

#### map to genes and annotate

In [21]:
#must give realpath to sample_participant_lookup
sos run  pipeline/gene_annotation.ipynb annotate_leafcutter_isoforms \
    --cwd output/normalize_impute \
    --intron_count output/leafcutter2/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz \
    --phenoFile output/normalize_impute/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.gz \
    --annotation_gtf /mnt/vast/hpc/csg/xqtl_workflow_testing/finalizing/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.gtf \
    --sample_participant_lookup  output/leafcutter2/sample_map_mod.txt.rnaseq \
    --map_stra region \
    --container  oras://ghcr.io/cumc/bioinfo_apptainer:latest --mem 100G


In [ ]:
#above command always complaining not enough memory, which should not be the case
import pandas as pd
import numpy as np
import qtl.io
from pathlib import Path
# Load data
tss_df = qtl.io.gtf_to_tss_bed("/mnt/vast/hpc/csg/xqtl_workflow_testing/finalizing/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.gtf")
bed_df = pd.read_csv("~/Work/leaf_cutter2/ROSMAP_AC_2024/output/normalize_impute/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.gz", sep='\t', skiprows=0)
bed_df.columns.values[0] = "#chr" # Temporary
sample_participant_lookup = Path("/mnt/vast/hpc/csg/rf2872/Work/leaf_cutter2/ROSMAP_AC_2024/output/leafcutter2/sample_map_mod.txt.rnaseq")
cluster2gene_dict = pd.read_csv("~/Work/leaf_cutter2/ROSMAP_AC_2024/output/normalize_impute/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz.leafcutter.clusters_to_genes.txt", sep='\t', index_col=0).to_dict()
cluster2gene_dict = cluster2gene_dict['genes']
print('    ** assigning introns to gene mapping(s)')
n = 0
gene_bed_df = []
group_s = {}
for _,r in bed_df.iterrows():
    s = r['ID'].split(':')
    cluster_id = s[0]+':'+s[3]
    if cluster_id in cluster2gene_dict:
        gene_ids = cluster2gene_dict[cluster_id].split(',')
        for g in gene_ids:
            gi = r['ID']+':'+g
            gene_bed_df.append(tss_df.loc[g, ['chr', 'start', 'end']].tolist() + [gi] + r.iloc[4:].tolist())
            group_s[gi] = g
    else:
        n += 1
        
        
if n > 0:
    print(f'    ** discarded {n} introns without a gene mapping')

print('  * writing BED files for QTL mapping')
gene_bed_df = pd.DataFrame(gene_bed_df, columns=bed_df.columns)
# sort by TSS
gene_bed_df = gene_bed_df.groupby('#chr', sort=False, group_keys=False).apply(lambda x: x.sort_values('start'))
#rename the samples if they named by file name (simply pick the first element with [.])
gene_bed_df.columns = list(gene_bed_df.columns[:4]) + [name.split('.')[0] if 'junc' in name else name for name in gene_bed_df.columns[4:]]
# change sample IDs to participant IDs
if sample_participant_lookup.is_file():
    sample_participant_lookup_s = pd.read_csv(sample_participant_lookup, sep="\t", index_col=0, dtype={0:str,1:str})
    #gene_bed_df.rename(columns=sample_participant_lookup_s.to_dict(), inplace=True)
    # Create a dictionary mapping from sample_id to participant_id
    column_mapping = dict(zip(sample_participant_lookup_s.index, sample_participant_lookup_s['participant_id']))
    # Get the column names to be replaced
    column_names = gene_bed_df.columns[4:]
    # Replace the column names using the mapping dictionary
    #new_column_names = [column_mapping.get(col, 'missing_data') for col in column_names]
    #it should overlap with genotype in downstream anyways
    new_column_names = [column_mapping.get(col, col) for col in column_names]
    gene_bed_df.rename(columns=dict(zip(column_names, new_column_names)), inplace=True)

gene_bed_df = gene_bed_df.drop_duplicates()
# qtl.io.write_bed(gene_bed_df, "~/Work/leaf_cutter2/ROSMAP_AC_2024/output/normalize_impute/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.formated.bed.gz")
uncompressed_path = "~/Work/leaf_cutter2/ROSMAP_AC_2024/output/normalize_impute/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.formated.bed"
gene_bed_df.to_csv(uncompressed_path, sep='\t', index=False)
bgzip_path = "/mnt/vast/hpc/homes/rf2872/software/htslib-1.9/bgzip"
compressed_path = uncompressed_path + ".gz"
bgzip_command = f"{bgzip_path} -c {uncompressed_path} > {compressed_path}"
#run below with bash 
tabix_path = "/mnt/vast/hpc/homes/rf2872/software/htslib-1.9/tabix"
tabix_command = f"{tabix_path}  -p bed  {compressed_path}"
#tabix -p bed ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.formated.bed.gz
subprocess.run(bgzip_command, shell=True, check=True)
subprocess.run(tabix_command, shell=True, check=True)

gene_bed_df[['start', 'end']] = gene_bed_df[['start', 'end']].astype(np.int32)
gene_bed_df[gene_bed_df.columns[4:]] = gene_bed_df[gene_bed_df.columns[4:]].astype(np.float32)
group_s_df =  pd.Series(group_s).sort_values().reset_index()
group_s_df.columns = ['ID', 'gene'] 
group_s_df.to_csv('~/Work/leaf_cutter2/ROSMAP_AC_2024/output/normalize_impute/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.formated..phenotype_group.txt', sep='\t', index=False, header=True)

#### partition

In [ ]:
sos run pipeline/phenotype_formatting.ipynb phenotype_by_chrom \
    --cwd output/data_preprocessing/phenotype_data/phenotype_by_chrom \
    --phenoFile output/normalize_impute/ROSMAP_AC_perind.counts.noise_by_intron.QCed_minc1_mins10.gz_raw_data.qqnorm.imputed.bed.formated.bed.gz \
    --chrom `for i in {1..22}; do echo chr$i; done` \
    --container /mnt/vast/hpc/csg/containers_xqtl/bioinfo.sif  --mem 40G -s force
